In [11]:
# Cell 1: Install dependencies
!pip install -q "av<16.0.0" faster-whisper gradio openai numpy librosa scikit-learn soundfile

# OPTIONAL (higher-accuracy speaker diarization, needs PyTorch + a free Hugging Face token):
# !pip install -q pyannote.audio
# then set HF_TOKEN in your environment and accept the terms at huggingface.co/pyannote/speaker-diarization-3.1


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# Cell 2: Configuration, models and LLM helpers (NO secrets stored in this notebook)
import os, re, json, time, getpass, tempfile, difflib, datetime
from collections import Counter
from typing import List, Dict, Optional
import numpy as np
from openai import OpenAI

# The key is read from the GROQ_API_KEY environment variable, or typed in once at runtime.
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key (not saved in the notebook): ")
client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1", timeout=60, max_retries=0)

# --- Model roles (3 models, 3 distinct jobs) ---
STT_BACKEND    = os.environ.get("STT_BACKEND", "auto")                     # auto = Groq Whisper first, local fallback | groq | local
GROQ_STT_MODEL = os.environ.get("GROQ_STT_MODEL", "whisper-large-v3-turbo") # speech-to-text (hosted, free tier)
STT_MODEL_SIZE = os.environ.get("STT_MODEL_SIZE", "large-v3-turbo")         # local faster-whisper model (fallback / offline)
LLM_REFINER    = os.environ.get("LLM_REFINER", "openai/gpt-oss-20b")       # LLM #1: transcript refinement (fast, small)
LLM_MINUTES    = os.environ.get("LLM_MINUTES", "openai/gpt-oss-120b")      # LLM #2: minutes / decisions / tasks (larger, better reasoning)

DOMAIN_GLOSSARY = [
    "CI/CD", "Kubernetes", "Docker", "PyTorch", "TensorFlow", "API", "OAuth", "PostgreSQL",
    "Microservices", "CapEx", "OpEx", "KPI", "OKR", "SLA", "ROI", "MVP", "GitHub", "AWS", "Azure", "GPU", "LLM", "EBITDA",
]

def check_models():
    """Warn early if a configured model is not available on this Groq account."""
    try:
        available = {m.id for m in client.models.list().data}
    except Exception as e:
        print(f"[warn] Could not list Groq models ({e}). Continuing.")
        return
    for name in (LLM_REFINER, LLM_MINUTES) + ((GROQ_STT_MODEL,) if STT_BACKEND != "local" else ()):
        if name not in available:
            print(f"[warn] Model '{name}' is not available. Pick one of: {sorted(available)}")
    if LLM_REFINER == LLM_MINUTES:
        print("[warn] Both LLM roles use the same model. The PS wants two distinct roles - use two different models.")
check_models()

_REASONING_OK = True
def llm_call(model, system, user, max_tokens=2000, json_mode=True, retries=4):
    """One place for all LLM calls: temperature 0, JSON mode, retry/backoff on rate limits."""
    global _REASONING_OK
    last = None
    for attempt in range(retries):
        kwargs = dict(model=model, temperature=0.0, max_tokens=max_tokens,
                      messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        if _REASONING_OK and model.startswith("openai/gpt-oss"):
            kwargs["extra_body"] = {"reasoning_effort": "low"}   # keeps reasoning tokens from eating the output budget
        try:
            r = client.chat.completions.create(**kwargs)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            last = e
            msg = str(e).lower()
            if "reasoning" in msg:
                _REASONING_OK = False; continue
            if any(k in msg for k in ("429", "rate", "timeout", "timed out", "connection", "503", "overloaded")):
                time.sleep(2 ** (attempt + 1)); continue
            if "json" in msg and json_mode:
                json_mode = False; continue
            break
    cause = repr(last.__cause__) if last is not None and last.__cause__ else ""
    hint = " (network problem: check internet/VPN/firewall/proxy - see the cause above)" if "connection" in str(last).lower() else ""
    raise RuntimeError(f"LLM call to '{model}' failed: {last} {cause}{hint}")

def parse_json(text: str) -> dict:
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        return json.loads(t)
    except Exception:
        a, b = t.find("{"), t.rfind("}")
        if a != -1 and b > a:
            return json.loads(t[a:b + 1])
        raise ValueError("The model did not return valid JSON.")

In [13]:
# Cell 3: Stage 1 - audio validation + speech-to-text (Groq Whisper first, local faster-whisper fallback)
import io
import soundfile as sf

ALLOWED_EXT = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac", ".wma", ".mp4", ".webm", ".opus"}
SR = 16000
STT_CHUNK_SEC = 600          # long audio is sent to Groq in ~10 minute pieces (keeps each upload small)
STT_INFO = {"name": ""}      # records which STT model actually produced the transcript
_stt_model = None

def get_stt_model():
    global _stt_model
    if _stt_model is None:
        from faster_whisper import WhisperModel
        print(f"Loading local faster-whisper ({STT_MODEL_SIZE}) - the first run downloads the model...")
        _stt_model = WhisperModel(STT_MODEL_SIZE, device="auto", compute_type="int8")
    return _stt_model

def load_audio(path: str) -> np.ndarray:
    """Validate the file and decode it ONCE to 16 kHz mono (reused by transcription and diarization)."""
    if not path or not os.path.exists(path):
        raise ValueError("No audio file found. Please upload a meeting recording.")
    ext = os.path.splitext(path)[1].lower()
    if ext not in ALLOWED_EXT:
        raise ValueError(f"Unsupported file type '{ext}'. Please upload one of: {', '.join(sorted(ALLOWED_EXT))}.")
    if os.path.getsize(path) == 0:
        raise ValueError("The uploaded audio file is empty (0 bytes).")
    from faster_whisper.audio import decode_audio
    try:
        audio = np.asarray(decode_audio(path, sampling_rate=SR), dtype=np.float32)
    except Exception as e:
        raise ValueError("The audio file could not be read. It may be corrupted or not real audio.") from e
    if audio.size < SR:
        raise ValueError("The recording is shorter than 1 second.")
    if float(np.abs(audio).max()) < 1e-3:
        raise ValueError("The recording appears to be silent.")
    return audio

def _split_points(audio: np.ndarray, chunk_sec: int) -> List[int]:
    """Cut long audio at the quietest moment near each chunk boundary so words are not split in half."""
    n, step = len(audio), chunk_sec * SR
    pts = [0]
    while n - pts[-1] > step * 1.15:                       # avoid a tiny last chunk
        target = pts[-1] + step
        w = min(20 * SR, step // 3)
        lo, hi = max(pts[-1] + SR, target - w), min(target + w, n)
        frame = int(0.1 * SR)
        seg = audio[lo:hi]
        m = len(seg) // frame
        rms = np.sqrt((seg[: m * frame].reshape(m, frame) ** 2).mean(axis=1))
        cut = lo + int(np.argmin(rms)) * frame
        if n - cut < 5 * SR:                                 # never leave a tiny last chunk
            break
        pts.append(cut)
    pts.append(n)
    return pts

def _groq_transcribe(audio: np.ndarray, glossary: List[str]) -> List[dict]:
    prompt = "Technical meeting. Terms: " + ", ".join(glossary[:40]) + "."     # biases spelling of domain terms
    pts = _split_points(audio, STT_CHUNK_SEC)
    out = []
    for a, b in zip(pts[:-1], pts[1:]):
        buf = io.BytesIO()
        sf.write(buf, audio[a:b], SR, format="FLAC", subtype="PCM_16")        # lossless + small
        payload = buf.getvalue()
        resp = None
        for attempt in range(3):
            try:
                resp = client.with_options(timeout=300).audio.transcriptions.create(
                    file=("audio.flac", payload), model=GROQ_STT_MODEL, language="en", temperature=0,
                    response_format="verbose_json", timestamp_granularities=["segment"], prompt=prompt)
                break
            except Exception as e:
                msg = str(e).lower()
                if attempt < 2 and any(k in msg for k in ("429", "rate", "timeout", "timed out", "connection", "503")):
                    time.sleep(2 ** (attempt + 2)); continue
                raise
        data = resp if isinstance(resp, dict) else resp.model_dump()
        offset = a / SR
        for s in data.get("segments") or []:
            text = (s.get("text") or "").strip()
            if not text:
                continue
            # standard Whisper heuristic: drop likely hallucinations over silence/noise
            if (s.get("no_speech_prob") or 0) > 0.6 and (s.get("avg_logprob") or 0) < -1.0:
                continue
            out.append({"start": offset + float(s["start"]), "end": offset + float(s["end"]), "text": text})
    return out

def _local_transcribe(audio: np.ndarray, glossary: List[str]) -> List[dict]:
    model = get_stt_model()
    prompt = "Technical meeting. Terms: " + ", ".join(glossary[:40]) + "."
    segments, _ = model.transcribe(
        audio, beam_size=5, language="en", vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
        condition_on_previous_text=False,   # stops Whisper repeating/looping on long audio
        initial_prompt=prompt,
    )
    return [{"start": float(s.start), "end": float(s.end), "text": s.text.strip()} for s in segments if s.text.strip()]

def run_stage1_transcribe(audio: np.ndarray, glossary: List[str]) -> List[dict]:
    out = None
    if STT_BACKEND in ("auto", "groq"):
        try:
            out = _groq_transcribe(audio, glossary)
            STT_INFO["name"] = f"Groq {GROQ_STT_MODEL}"
        except Exception as e:
            print(f"[warn] Groq Whisper failed: {e}")
            if STT_BACKEND == "groq":
                raise ValueError(f"Groq speech-to-text failed: {e}") from e
            print("[warn] Falling back to local faster-whisper.")
    if out is None:
        out = _local_transcribe(audio, glossary)
        STT_INFO["name"] = f"faster-whisper {STT_MODEL_SIZE} (local)"
    if not out:
        raise ValueError("No intelligible speech was detected in the recording.")
    return out

In [14]:
# Cell 4: Speaker diarization ("who said what")
def fmt_ts(sec: float) -> str:
    h, rem = divmod(int(sec), 3600); m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def _pyannote_turns(audio: np.ndarray, num_speakers: Optional[int]):
    """Best accuracy. Used only if pyannote + PyTorch are installed and HF_TOKEN is set."""
    token = os.environ.get("HF_TOKEN")
    if not token:
        return None
    try:
        import torch
        from pyannote.audio import Pipeline
    except Exception:
        return None
    try:
        name = "pyannote/speaker-diarization-3.1"
        try:
            pipe = Pipeline.from_pretrained(name, token=token)
        except TypeError:
            pipe = Pipeline.from_pretrained(name, use_auth_token=token)
        if torch.cuda.is_available():
            pipe.to(torch.device("cuda"))
        kw = {"num_speakers": num_speakers} if num_speakers else {}
        out = pipe({"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": SR}, **kw)
        ann = getattr(out, "speaker_diarization", out)
        return [(t.start, t.end, spk) for t, _, spk in ann.itertracks(yield_label=True)]
    except Exception as e:
        print(f"[warn] pyannote diarization failed ({e}); using the lightweight fallback.")
        return None

def _assign_by_overlap(segments, turns):
    labels = []
    for s in segments:
        scores = {}
        for a, b, spk in turns:
            ov = min(s["end"], b) - max(s["start"], a)
            if ov > 0:
                scores[spk] = scores.get(spk, 0.0) + ov
        if scores:
            labels.append(max(scores, key=scores.get))
        else:  # no overlap: take the nearest speaker turn
            mid = (s["start"] + s["end"]) / 2
            labels.append(min(turns, key=lambda t: min(abs(mid - t[0]), abs(mid - t[1])))[2])
    return labels

def _voice_features(clip: np.ndarray) -> np.ndarray:
    import librosa
    mfcc = librosa.feature.mfcc(y=clip, sr=SR, n_mfcc=20, n_fft=512, hop_length=160)
    f0 = librosa.yin(clip, fmin=65, fmax=400, sr=SR, frame_length=1024)
    cent = librosa.feature.spectral_centroid(y=clip, sr=SR, n_fft=512, hop_length=160)
    return np.concatenate([mfcc.mean(1), mfcc.std(1), [np.median(f0), cent.mean()]])

def _mfcc_labels(audio, segments, num_speakers, max_speakers=4):
    """Lightweight fallback (no PyTorch, no token): voice features + clustering. Less accurate than pyannote."""
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler
    feats, idx = [], []
    for i, s in enumerate(segments):
        clip = audio[int(s["start"] * SR): int(s["end"] * SR)]
        if len(clip) >= int(0.8 * SR):          # very short clips give unreliable voice features
            feats.append(_voice_features(clip)); idx.append(i)
    n = len(feats)
    labels = [0] * len(segments)
    if n < 2:
        return labels
    X = StandardScaler().fit_transform(np.array(feats))
    k = None
    if num_speakers and num_speakers > 1:
        k = min(num_speakers, n)
    elif num_speakers == 1:
        return labels
    elif n >= 3:   # auto-detect the speaker count with the silhouette score
        best, best_k = -1.0, 1
        for kk in range(2, min(max_speakers, n - 1) + 1):
            cand = AgglomerativeClustering(n_clusters=kk, linkage="ward").fit_predict(X)
            if len(set(cand)) > 1:
                sc = silhouette_score(X, cand)
                if sc > best:
                    best, best_k = sc, kk
        k = best_k if best >= 0.15 else 1
    if not k or k < 2:
        return labels
    cl = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
    for j, i in enumerate(idx):
        labels[i] = int(cl[j]) 
    # short segments inherit the previous (or next) confidently-labelled speaker
    known = set(idx)
    for i in range(len(segments)):
        if i not in known:
            prev = next((labels[j] for j in range(i - 1, -1, -1) if j in known), None)
            nxt = next((labels[j] for j in range(i + 1, len(segments)) if j in known), 0)
            labels[i] = prev if prev is not None else nxt
    return labels

def diarize(audio, segments, num_speakers: Optional[int] = None):
    """Returns ('Speaker N' label per segment, method name). Speakers are numbered by first appearance."""
    turns = _pyannote_turns(audio, num_speakers)
    if turns:
        raw, method = _assign_by_overlap(segments, turns), "pyannote 3.1"
    else:
        raw, method = _mfcc_labels(audio, segments, num_speakers), "MFCC+pitch clustering (lightweight)"
    order, labels = {}, []
    for r in raw:
        order.setdefault(r, len(order) + 1)
        labels.append(f"Speaker {order[r]}")
    return labels, method

def build_turns(segments, labels, max_gap=2.0):
    """Merge consecutive segments from the same speaker into one speaking turn."""
    turns = []
    for s, spk in zip(segments, labels):
        if turns and turns[-1]["speaker"] == spk and s["start"] - turns[-1]["end"] <= max_gap:
            turns[-1]["end"] = s["end"]; turns[-1]["text"] += " " + s["text"]
        else:
            turns.append({"id": len(turns), "speaker": spk, "start": s["start"], "end": s["end"], "text": s["text"]})
    return turns

def transcript_text(turns) -> str:
    return "\n".join(f"[{fmt_ts(t['start'])}] {t['speaker']}: {t['text']}" for t in turns)

In [15]:
# Cell 5: Stage 2 - domain-aware refinement (LLM #1) with safety guards
REFINER_PROMPT = """You correct speech-to-text recognition errors in meeting transcript turns.
Domain glossary (prefer these spellings when a phrase sounds like one of them): [{{GLOSSARY}}]

INPUT:  {"turns": [{"id": 0, "text": "..."}, ...]}
OUTPUT: {"turns": [{"id": 0, "text": "..."}, ...]}  - the SAME ids, the SAME number of turns, same order.

RULES:
1. Only change words that are clearly mis-recognitions of technical terms, acronyms, product names or jargon.
   Example: "cooper netties" -> "Kubernetes", "sea eye see dee" -> "CI/CD".
2. You may fix capitalisation and punctuation. Do NOT rephrase, summarise, translate, add or remove sentences.
3. Keep every name, number, date, negation (not, no, never, can't, won't ...) and commitment EXACTLY as spoken.
4. If you are unsure, leave the text unchanged.
5. Never merge or split turns. Return ONLY the JSON object."""

NEG_WORDS = {"not", "no", "never", "cannot", "without", "nothing", "none", "nobody", "neither", "nor"}

def _negations(t: str) -> int:
    toks = re.findall(r"[a-z']+", t.lower())
    return sum(1 for w in toks if w in NEG_WORDS or w.endswith("n't"))

def _numbers(t: str) -> Counter:
    return Counter(re.findall(r"\d+(?:[.,]\d+)?", t))

def guard_ok(raw: str, new: str) -> bool:
    """Reject a refinement that changed numbers, negations, or rewrote the sentence."""
    if not new.strip():
        return False
    if _numbers(raw) != _numbers(new) or _negations(raw) != _negations(new):
        return False
    if len(raw.split()) > 3:
        ratio = len(new) / max(1, len(raw))
        if ratio < 0.75 or ratio > 1.35:
            return False
        if difflib.SequenceMatcher(None, raw.lower(), new.lower()).ratio() < 0.6:
            return False
    return True

def word_diff(a: str, b: str):
    aw, bw = a.split(), b.split()
    sm = difflib.SequenceMatcher(None, aw, bw, autojunk=False)
    return [(" ".join(aw[i1:i2]), " ".join(bw[j1:j2])) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"]

def refine_turns(turns, glossary):
    system = REFINER_PROMPT.replace("{{GLOSSARY}}", ", ".join(glossary))
    batches, cur, size = [], [], 0
    for t in turns:
        if cur and (size + len(t["text"]) > 3500 or len(cur) >= 25):
            batches.append(cur); cur, size = [], 0
        cur.append(t); size += len(t["text"])
    if cur:
        batches.append(cur)

    new_text, stats = {}, {"turns": len(turns), "changed": 0, "reverted_by_guard": 0, "batch_errors": 0}
    for batch in batches:
        payload = json.dumps({"turns": [{"id": t["id"], "text": t["text"]} for t in batch]}, ensure_ascii=False)
        try:
            data = parse_json(llm_call(LLM_REFINER, system, payload, max_tokens=min(3500, len(payload) // 2 + 600)))
            returned = {int(x["id"]): str(x["text"]).strip() for x in data.get("turns", []) if "id" in x and "text" in x}
        except Exception as e:
            print(f"[warn] refinement batch failed ({e}); keeping raw text for this batch.")
            stats["batch_errors"] += 1
            returned = {}
        for t in batch:
            cand = returned.get(t["id"])
            if cand is None:
                new_text[t["id"]] = t["text"]
            elif guard_ok(t["text"], cand):
                new_text[t["id"]] = cand
            else:
                new_text[t["id"]] = t["text"]; stats["reverted_by_guard"] += 1

    refined, changes = [], []
    for t in turns:
        r = dict(t); r["text"] = new_text[t["id"]]; refined.append(r)
        if r["text"] != t["text"]:
            stats["changed"] += 1
            diffs = word_diff(t["text"], r["text"])
            if diffs:
                changes.append({"time": fmt_ts(t["start"]), "speaker": t["speaker"], "changes": [[a, b] for a, b in diffs]})
    return refined, stats, changes

def corrections_markdown(changes) -> str:
    if not changes:
        return "_No terminology corrections were needed or applied._"
    lines = ["| Time | Speaker | Raw (heard) | Refined (corrected) |", "|---|---|---|---|"]
    for c in changes:
        for a, b in c["changes"]:
            lines.append(f"| {c['time']} | {c['speaker']} | {a.replace('|','/') or '-'} | {b.replace('|','/') or '-'} |")
    return "\n".join(lines)

In [16]:
# Cell 6: Stage 3 - minutes, decisions, proposals, action items (LLM #2) + grounding verification
MINUTES_PROMPT = """You are a meticulous meeting secretary. Work ONLY from the transcript. Each line looks like "[mm:ss] Speaker: text".
Return ONE JSON object:
{
  "summary": "3-5 sentence summary",
  "minutes": [{"topic": "...", "points": ["..."]}],
  "decisions": [{"decision": "...", "decided_by": "speaker or unspecified", "evidence": "verbatim quote"}],
  "open_proposals": [{"proposal": "...", "proposed_by": "speaker or unspecified", "evidence": "verbatim quote"}],
  "action_items": [{"task": "...", "owner": "...", "deadline": "...", "evidence": "verbatim quote"}]
}
RULES:
1. DECISION = something explicitly agreed, approved or confirmed ("let's go with", "agreed", "approved", "we will"). Suggestions, ideas, questions and "maybe we should" are PROPOSALS: put them in open_proposals, never in decisions.
2. ACTION ITEM = a concrete task that someone clearly committed to or was assigned. Vague wishes are not action items.
3. owner = a person only if they were named as responsible, or a speaker volunteered ("I'll do it"; then use that speaker's label or name). Otherwise "unspecified". NEVER guess.
4. deadline = copy the time phrase exactly as spoken (e.g. "by Friday"). If none was stated: "unspecified". Never invent or convert dates.
5. evidence must be copied WORD FOR WORD from the transcript (one or two sentences at most).
6. If the audio is a lecture or talk rather than a meeting, capture key points in minutes and return [] for decisions, open_proposals and action_items.
7. Do not invent anything. Omit rather than guess. At most 8 minute topics. Return ONLY the JSON object."""

UNSPEC = {"", "unspecified", "n/a", "na", "none", "unknown", "not specified", "not stated", "tbd", "null", "-"}
MAX_CHARS_PER_CHUNK = 6000

def _s(v, default="") -> str:
    return default if v is None else (str(v).strip() or default)

def _unspec(v) -> str:
    s = _s(v)
    return "unspecified" if s.lower() in UNSPEC else s

def _norm(t: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]+", " ", t.lower())).strip()

def _normalize_extraction(d: dict) -> dict:
    out = {"summary": _s(d.get("summary")), "minutes": [], "decisions": [], "open_proposals": [], "action_items": []}
    for m in d.get("minutes") or []:
        if isinstance(m, dict):
            pts = m.get("points") or []
            pts = [pts] if isinstance(pts, str) else pts
            out["minutes"].append({"topic": _s(m.get("topic"), "General"), "points": [_s(p) for p in pts if _s(p)]})
    for x in d.get("decisions") or []:
        if isinstance(x, dict) and _s(x.get("decision")):
            out["decisions"].append({"decision": _s(x["decision"]), "decided_by": _unspec(x.get("decided_by")), "evidence": _s(x.get("evidence"))})
    for x in d.get("open_proposals") or []:
        if isinstance(x, dict) and _s(x.get("proposal")):
            out["open_proposals"].append({"proposal": _s(x["proposal"]), "proposed_by": _unspec(x.get("proposed_by")), "evidence": _s(x.get("evidence"))})
    for x in d.get("action_items") or []:
        if isinstance(x, dict) and _s(x.get("task")):
            out["action_items"].append({"task": _s(x["task"]), "owner": _unspec(x.get("owner")), "deadline": _unspec(x.get("deadline")), "evidence": _s(x.get("evidence"))})
    return out

def _dedupe(items, key):
    seen, out = set(), []
    for it in items:
        k = _norm(it[key])[:80]
        if k not in seen:
            seen.add(k); out.append(it)
    return out

def _is_grounded(evidence: str, turn_norms: List[str], full_norm: str) -> bool:
    ev = _norm(evidence)
    if len(ev.split()) < 3:
        return False
    if ev in full_norm:
        return True
    for tn in turn_norms:   # tolerate tiny wording drift (>=85% of the quote found contiguously)
        m = difflib.SequenceMatcher(None, ev, tn, autojunk=False).find_longest_match(0, len(ev), 0, len(tn))
        if m.size / len(ev) >= 0.85:
            return True
    return False

def verify_extraction(ext: dict, turns) -> dict:
    """Anti-hallucination layer: every decision/task must quote the transcript; owners/deadlines must really appear in it."""
    turn_norms = [_norm(t["text"]) for t in turns]
    full_norm = " ".join(turn_norms)
    speakers = {_norm(t["speaker"]) for t in turns}
    q = {"dropped_not_grounded": 0, "owners_reset_to_unspecified": 0, "deadlines_reset_to_unspecified": 0}

    def keep(items):
        kept = []
        for it in items:
            if _is_grounded(it.get("evidence", ""), turn_norms, full_norm):
                kept.append(it)
            else:
                q["dropped_not_grounded"] += 1
        return kept

    ext["decisions"] = keep(ext["decisions"])
    ext["open_proposals"] = keep(ext["open_proposals"])
    ext["action_items"] = keep(ext["action_items"])
    for a in ext["action_items"]:
        if a["owner"] != "unspecified" and _norm(a["owner"]) not in speakers and _norm(a["owner"]) not in full_norm:
            a["owner"] = "unspecified"; q["owners_reset_to_unspecified"] += 1
        if a["deadline"] != "unspecified" and _norm(a["deadline"]) not in full_norm:
            a["deadline"] = "unspecified"; q["deadlines_reset_to_unspecified"] += 1
    q.update({"decisions": len(ext["decisions"]), "open_proposals": len(ext["open_proposals"]), "action_items": len(ext["action_items"])})
    ext["quality_report"] = q
    return ext

def run_stage3_documentation(turns) -> dict:
    lines = transcript_text(turns).split("\n")
    chunks, cur, size = [], [], 0
    for ln in lines:
        if cur and size + len(ln) > MAX_CHARS_PER_CHUNK:
            chunks.append("\n".join(cur)); cur, size = [], 0
        cur.append(ln); size += len(ln) + 1
    if cur:
        chunks.append("\n".join(cur))

    parts = []
    for c in chunks:
        raw = llm_call(LLM_MINUTES, MINUTES_PROMPT, "Transcript:\n" + c, max_tokens=3000)
        parts.append(_normalize_extraction(parse_json(raw)))

    merged = {"summary": parts[0]["summary"], "minutes": [], "decisions": [], "open_proposals": [], "action_items": []}
    for p in parts:
        for k in ("minutes", "decisions", "open_proposals", "action_items"):
            merged[k].extend(p[k])
    if len(parts) > 1:   # map-reduce: combine chunk summaries into one
        try:
            joined = "\n".join(f"Part {i+1}: {p['summary']}" for i, p in enumerate(parts))
            r = parse_json(llm_call(LLM_MINUTES, 'Combine these partial meeting summaries into one coherent 3-5 sentence summary. Do not add facts. Return JSON {"summary": "..."}.', joined, max_tokens=1000))
            merged["summary"] = _s(r.get("summary")) or " ".join(p["summary"] for p in parts)
        except Exception:
            merged["summary"] = " ".join(p["summary"] for p in parts)
    merged["minutes"] = _dedupe(merged["minutes"], "topic")
    merged["decisions"] = _dedupe(merged["decisions"], "decision")
    merged["open_proposals"] = _dedupe(merged["open_proposals"], "proposal")
    merged["action_items"] = _dedupe(merged["action_items"], "task")
    return verify_extraction(merged, turns)

In [17]:
# Cell 7: Rendering, speaker renaming, outputs and the end-to-end orchestrator
def render_markdown(rec: dict) -> str:
    meta = rec["meta"]
    md = ["# Meeting Record", "", "## Summary", rec.get("summary") or "_No summary generated._", "",
          f"**Participants detected:** {', '.join(meta['speakers']) or 'unknown'}  ", f"**Duration:** {meta['duration']}", "", "## Minutes"]
    if rec["minutes"]:
        for m in rec["minutes"]:
            md.append(f"### {m['topic']}")
            md += [f"- {p}" for p in m["points"]]
    else:
        md.append("_No structured minutes._")
    md += ["", "## Key Decisions"]
    if rec["decisions"]:
        for d in rec["decisions"]:
            md.append(f"- **{d['decision']}** (decided by: {d['decided_by']})")
            md.append(f"  > {d['evidence']}")
    else:
        md.append("_No explicit decisions were made in this recording._")
    md += ["", "## Proposals / Open Items (discussed but NOT agreed)"]
    if rec["open_proposals"]:
        for p in rec["open_proposals"]:
            md.append(f"- {p['proposal']} (proposed by: {p['proposed_by']})")
            md.append(f"  > {p['evidence']}")
    else:
        md.append("_None._")
    md += ["", "## Action Items"]
    if rec["action_items"]:
        for a in rec["action_items"]:
            md.append(f"- **{a['task']}** - Owner: {a['owner']} | Deadline: {a['deadline']}")
            md.append(f"  > {a['evidence']}")
    else:
        md.append("_No actionable tasks were assigned._")
    q = rec["quality_report"]
    md += ["", "---", f"_Verification: every decision/task above is backed by a quote found in the transcript. "
           f"Dropped as unsupported: {q['dropped_not_grounded']}. Owners reset to unspecified: {q['owners_reset_to_unspecified']}. "
           f"Deadlines reset to unspecified: {q['deadlines_reset_to_unspecified']}._"]
    return "\n".join(md)

def parse_name_map(s: str) -> Dict[str, str]:
    m = {}
    for part in re.split(r"[,;\n]+", s or ""):
        if "=" in part:
            k, v = part.split("=", 1)
            nums = re.findall(r"\d+", k)
            if re.fullmatch(r"(?i)\s*speaker\s*\d+\s*", k) and v.strip() and nums:
                m[f"Speaker {int(nums[0])}"] = v.strip()
    return m

def rename_speakers(rec: dict, mapping: Dict[str, str]) -> dict:
    if not mapping:
        return rec
    esc = {k: json.dumps(v, ensure_ascii=False)[1:-1] for k, v in mapping.items()}
    text = re.sub(r"\bSpeaker \d+\b", lambda x: esc.get(x.group(0), x.group(0)), json.dumps(rec, ensure_ascii=False))
    return json.loads(text)

def save_outputs(rec: dict):
    out_dir = tempfile.mkdtemp(prefix="meeting_")
    paths = {
        "raw_transcript.txt": transcript_text(rec["transcript"]["raw"]),
        "refined_transcript.txt": transcript_text(rec["transcript"]["refined"]),
        "meeting_record.md": render_markdown(rec),
        "meeting_record.json": json.dumps(rec, indent=2, ensure_ascii=False),
    }
    files = []
    for name, content in paths.items():
        p = os.path.join(out_dir, name)
        with open(p, "w", encoding="utf-8") as f:
            f.write(content)
        files.append(p)
    return files

def _views(rec):
    return (transcript_text(rec["transcript"]["raw"]), transcript_text(rec["transcript"]["refined"]),
            corrections_markdown(rec["transcript"]["corrections"]), render_markdown(rec))

def run_pipeline(audio_path, num_speakers, extra_terms, speaker_names):
    """Generator: streams each stage's result to the UI as soon as it is ready."""
    raw_text = refined_text = corr = md = ""
    def out(status, files=None, state=None):
        return (raw_text, refined_text, corr, md, files, status, state)
    try:
        t0 = time.time()
        glossary = DOMAIN_GLOSSARY + [t.strip() for t in re.split(r"[,\n]+", extra_terms or "") if t.strip()]
        yield out("Status: validating and decoding audio...")
        audio = load_audio(audio_path)
        duration = len(audio) / SR

        yield out("Status: Stage 1/3 - transcribing speech...")
        segs = run_stage1_transcribe(audio, glossary)
        yield out("Status: Stage 1/3 - identifying speakers...")
        labels, method = diarize(audio, segs, int(num_speakers) if num_speakers else None)
        raw_turns = build_turns(segs, labels)
        names = parse_name_map(speaker_names)
        raw_text = transcript_text(rename_speakers({"t": raw_turns}, names)["t"])
        yield out("Status: Stage 2/3 - refining domain terminology (LLM #1)...")

        refined_turns, rstats, changes = refine_turns(raw_turns, glossary)
        refined_text = transcript_text(rename_speakers({"t": refined_turns}, names)["t"])
        corr = corrections_markdown(changes)
        yield out("Status: Stage 3/3 - writing minutes, decisions and tasks (LLM #2)...")

        core = run_stage3_documentation(refined_turns)
        rec = {
            "meta": {"stt_model": STT_INFO["name"], "refiner_model": LLM_REFINER, "minutes_model": LLM_MINUTES,
                     "diarization": method, "speakers": sorted({t["speaker"] for t in raw_turns}, key=lambda s: int(s.split()[-1])),
                     "duration": fmt_ts(duration), "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
                     "refinement_stats": rstats},
            **core,
            "transcript": {"raw": raw_turns, "refined": refined_turns, "corrections": changes},
        }
        rec = rename_speakers(rec, names)
        raw_text, refined_text, corr, md = _views(rec)
        files = save_outputs(rec)
        yield out(f"Status: Complete in {time.time() - t0:.0f}s | STT: {STT_INFO['name']} | speakers: {len(rec['meta']['speakers'])} | "
                  f"corrections applied: {rstats['changed']} (guard reverted {rstats['reverted_by_guard']})"
                  + (f" | WARNING: refinement failed for {rstats['batch_errors']} batch(es) - check the LLM_REFINER model name" if rstats["batch_errors"] else ""), files, rec)
    except Exception as err:
        yield out(f"Status: FAILED - {err}")

def apply_names_ui(state, speaker_names):
    """Rename 'Speaker N' everywhere after the run - no LLM call needed."""
    if not state:
        return ("", "", "", "", None, "Status: run the pipeline first.", None)
    rec = rename_speakers(state, parse_name_map(speaker_names))
    raw, ref, corr, md = _views(rec)
    return (raw, ref, corr, md, save_outputs(rec), "Status: speaker names applied.", rec)

In [18]:
# Cell 8: Interactive web UI
import gradio as gr

with gr.Blocks(title="NoScribe") as demo:
    gr.Markdown("# NoScribe\nUpload a meeting recording and click **Process Recording** once. "
                "Speech-to-text, speaker identification, terminology refinement and meeting documentation run automatically.")
    state = gr.State(None)
    with gr.Row():
        audio_in = gr.Audio(sources=["upload"], type="filepath", label="Meeting audio (.mp3, .wav, .m4a, .flac, .ogg ...)")
        with gr.Column():
            num_spk = gr.Number(label="Number of speakers (0 = auto-detect)", value=0, precision=0)
            extra_terms = gr.Textbox(label="Extra domain terms (comma separated, optional)", placeholder="Terraform, Snowflake, Q3 OKRs")
            spk_names = gr.Textbox(label="Speaker names (optional)", placeholder="Speaker 1=Priya, Speaker 2=Alex")
    with gr.Row():
        run_btn = gr.Button("Process Recording", variant="primary")
        rename_btn = gr.Button("Apply speaker names to results")
    status = gr.Textbox(value="Status: Ready", label="Pipeline status", interactive=False)
    with gr.Tabs():
        with gr.Tab("1. Raw transcript (speech-to-text)"):
            raw_box = gr.Textbox(label="Raw transcript with speakers", lines=16, interactive=False)
        with gr.Tab("2. Refined transcript (LLM #1)"):
            ref_box = gr.Textbox(label="Domain-refined transcript with speakers", lines=16, interactive=False)
        with gr.Tab("Corrections (raw vs refined)"):
            corr_md = gr.Markdown()
        with gr.Tab("3. Meeting record (LLM #2)"):
            rec_md = gr.Markdown()
    files_out = gr.File(label="Downloads: raw transcript, refined transcript, minutes (.md), structured record (.json)", file_count="multiple")

    outs = [raw_box, ref_box, corr_md, rec_md, files_out, status, state]
    run_btn.click(run_pipeline, inputs=[audio_in, num_spk, extra_terms, spk_names], outputs=outs)
    rename_btn.click(apply_names_ui, inputs=[state, spk_names], outputs=outs)

demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
